# 🎬 Antigravity 4K Motion Graphics Batch Renderer (Real WebGL Canvas Engine)
Studio Otomatis Produksi Video Loop 4K (3840x2160, 30 FPS, Seamless Loop) untuk Adobe Stock & Freepik.

### ⚡ Fitur Auto-Download Per-Video:
- Setiap 1 video selesai di-render, video tersebut **LANGSUNG OTOMATIS TER-DOWNLOAD** ke laptop/PC Anda seketika itu juga.
- Jika di tengah jalan Colab terputus/disconnect, **video yang sudah selesai sebelumnya TIDAK AKAN HILANG** karena sudah tersimpan aman di folder Download komputer Anda!
- Tetap disediakan opsi download ZIP di akhir jika ingin mengunduh seluruhnya dalam satu paket.

## ⚙️ Step 1: Install Official Google Chrome & WebGL Headless Environment

In [ ]:
import os, subprocess, shutil
os.chdir('/content')

# 1. Download dan Install Official Google Chrome Stable Binary
print("⏳ Installing Official Google Chrome Stable GPU...")
!wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i /tmp/chrome.deb > /dev/null 2>&1 || apt-get install -fy > /dev/null 2>&1
!apt-get install -y ffmpeg libgbm-dev libnss3 libasound2 zip > /dev/null 2>&1
!rm -f /tmp/chrome.deb

# 2. Clone Repository Web App dari GitHub
print("⏳ Cloning Studio Repository...")
if os.path.exists('/content/shadergradientpaper'):
    shutil.rmtree('/content/shadergradientpaper', ignore_errors=True)

!git clone https://github.com/consistmaker/shadergradientpaper.git /content/shadergradientpaper

# 3. Install NPM Dependencies & Build Production WebGL Studio
print("⏳ Building WebGL Production Bundles...")
%cd /content/shadergradientpaper
!npm install --legacy-peer-deps > /dev/null 2>&1
!npm install puppeteer-core > /dev/null 2>&1
!npm run build
%cd /content

print("\n✅ OFFICIAL GOOGLE CHROME INSTALLED & 4K WEBGL HEADLESS ENGINE READY!")

## 📥 Step 2: Masukkan Recipe JSON dari Live Previewer
Paste JSON hasil tombol **Export Batch** dari web Live Previewer di bawah ini.

In [ ]:
import json
import os

# Paste JSON dari Live Previewer di sini:
RECIPE_JSON = '''
{
  "metadata": {
    "targetResolution": "3840x2160 (4K UHD)",
    "targetFps": 30,
    "loopDurationSeconds": 10,
    "isSeamlessLoop": true,
    "batchMode": "manual_queue"
  },
  "manualQueueList": [
    {
      "index": 1,
      "id": "item_1",
      "name": "Paper: mesh-gradient (#e0eaff)",
      "engine": "paper",
      "config": {
        "shaderType": "mesh-gradient",
        "color1": "#e0eaff",
        "color2": "#241d9a",
        "color3": "#f75092",
        "color4": "#9f50d3",
        "speed": 1.0,
        "distortion": 0.8,
        "swirl": 0.1
      }
    }
  ],
  "totalVideosInQueue": 1
}
'''

with open('/content/render_recipe.json', 'w') as f:
    f.write(RECIPE_JSON.strip())

recipe = json.loads(RECIPE_JSON)
batch_mode = recipe['metadata'].get('batchMode', 'manual_queue')
print(f"🎯 Batch Mode: {batch_mode.upper()}")
print(f"🎬 Target Specs: {recipe['metadata']['targetResolution']} @ {recipe['metadata']['targetFps']} FPS ({recipe['metadata']['loopDurationSeconds']}s Loop)")
print("✅ Recipe JSON Saved to /content/render_recipe.json")

## 🚀 Step 3: Eksekusi Render 4K WebGL (Auto-Download Langsung Tiap Selesai 1 Video)

In [ ]:
import subprocess, time, os, glob
from google.colab import files

output_dir = '/content/output_4k_videos'
os.makedirs(output_dir, exist_ok=True)
downloaded_files = set()

print("🚀 Starting 4K WebGL Render Pipeline with Instant Per-Video Auto-Download...")

# Jalankan Node.js Headless Renderer di background subprocess
process = subprocess.Popen(
    ['node', '/content/shadergradientpaper/headless_renderer.cjs'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1
)

# Monitor output dan langsung download setiap kali ada file video .mp4 baru yang selesai
while True:
    line = process.stdout.readline()
    if line:
        print(line, end='')
        
        # Deteksi tanda video sukses di-render
        if '✅ Success 4K Render:' in line:
            time.sleep(1)
            current_videos = glob.glob(f"{output_dir}/*.mp4")
            for v_path in current_videos:
                if v_path not in downloaded_files and os.path.exists(v_path):
                    file_size = (os.path.getsize(v_path) / (1024 * 1024))
                    print(f"   📥 [AUTO-DOWNLOAD] Mengunduh langsung ke komputer: {os.path.basename(v_path)} ({file_size:.2f} MB)...")
                    try:
                        files.download(v_path)
                        downloaded_files.add(v_path)
                    except Exception as e:
                        print(f"   ⚠️ Browser download prompt triggered: {e}")
                        
    if process.poll() is not None:
        # Baca sisa output jika ada
        for remaining in process.stdout.readlines():
            print(remaining, end='')
        break

# Pastikan semua video yang belum ter-download langsung di-download
remaining_videos = glob.glob(f"{output_dir}/*.mp4")
for v_path in remaining_videos:
    if v_path not in downloaded_files:
        print(f"📥 [FINAL DOWNLOAD] Mengunduh: {os.path.basename(v_path)}...")
        files.download(v_path)
        downloaded_files.add(v_path)

print(f"\n🎉 SELESAI! Total {len(downloaded_files)} video 4K telah otomatis ter-download ke PC/Laptop Anda.")

## 📦 Step 4 (Opsional): Download Cadangan Sebagai 1 File ZIP

In [ ]:
import os
from google.colab import files

if os.path.exists('/content/output_4k_videos') and len(os.listdir('/content/output_4k_videos')) > 0:
    !cd /content && zip -r /content/4K_Motion_Graphics_Batch.zip output_4k_videos
    print("\n📦 Packaging Complete! Downloading backup ZIP file...")
    files.download('/content/4K_Motion_Graphics_Batch.zip')
else:
    print("❌ Tidak ada video di folder output.")